# Experiment Tracking & Reproducibility — Notebook

**Week 10 · CS 203 · Software Tools and Techniques for AI**

Prof. Nipun Batra · IIT Gandhinagar

This notebook covers:
1. PyTorch reproducibility (seeds, deterministic mode, DataLoader)
2. Multi-seed reporting
3. W&B experiment tracking

> Hyperparameter tuning (Grid, Random, Optuna) was covered in Week 7.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy matplotlib torch scikit-learn wandb

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")

## 1. PyTorch Reproducibility

In [ ]:
import torch
import torch.nn as nn
import random
import os

def set_seed_full(seed=42):
    """Complete seed function for PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

### Without vs with seeds

In [ ]:
# Without seeds: different every time
print("=== Without seeds ===")
for i in range(3):
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y = model(x)
    print(f"  Run {i+1}: first output = {y[0].item():.6f}")

# With seeds: identical every time
print("
=== With seeds ===")
for i in range(3):
    set_seed_full(42)
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y = model(x)
    print(f"  Run {i+1}: first output = {y[0].item():.6f}")

### DataLoader reproducibility

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print("=== DataLoader with worker seeding ===")
for i in range(3):
    set_seed_full(42)
    g = torch.Generator()
    g.manual_seed(42)
    dataset = TensorDataset(torch.randn(100, 10), torch.randint(0, 2, (100,)))
    loader = DataLoader(
        dataset, batch_size=16, shuffle=True,
        worker_init_fn=seed_worker, generator=g
    )
    first_batch = next(iter(loader))
    print(f"  Run {i+1}: first batch sum = {first_batch[0].sum().item():.4f}, "
          f"labels = {first_batch[1][:5].tolist()}")

## 2. Multi-Seed Reporting

Better than one deterministic run: report mean +/- std across seeds.

In [ ]:
results = []
for seed in [42, 123, 456, 789, 1024]:
    np.random.seed(seed)
    X_s = np.random.rand(500, 8)
    y_s = (X_s[:, 0] * 2 + X_s[:, 2] - X_s[:, 5] + np.random.randn(500) * 0.3 > 1).astype(int)
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    results.append(acc)
    print(f"  Seed {seed:>4d}: {acc:.4f}")

print(f"
Accuracy: {np.mean(results):.4f} +/- {np.std(results):.4f}")
print("This is more informative than a single deterministic run!")

## 3. W&B Integration (Optional)

Uncomment and run if you have a W&B account.

In [ ]:
# import wandb

# wandb.init(project="movie-predictor-demo", config={
#     "n_estimators": 100,
#     "max_depth": 10,
#     "seed": 42,
# })

# model = RandomForestClassifier(
#     n_estimators=wandb.config.n_estimators,
#     max_depth=wandb.config.max_depth,
#     random_state=wandb.config.seed,
# )

# np.random.seed(42)
# X = np.random.rand(1000, 8)
# y = (X[:, 0] * 2 + X[:, 2] - X[:, 5] + np.random.randn(1000) * 0.3 > 1).astype(int)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model.fit(X_train, y_train)
# train_acc = model.score(X_train, y_train)
# test_acc = model.score(X_test, y_test)

# wandb.log({"train_accuracy": train_acc, "test_accuracy": test_acc})

# for i, imp in enumerate(model.feature_importances_):
#     wandb.log({f"feature_{i}_importance": imp})

# print(f"Train: {train_acc:.4f}, Test: {test_acc:.4f}")
# print(f"View at: {wandb.run.url}")
# wandb.finish()

## Summary

**Key takeaways:**
- PyTorch reproducibility requires seeds + deterministic algorithms + DataLoader seeding
- Report results over multiple seeds for reliability
- W&B automates experiment tracking (params, metrics, code, artifacts)
- MLflow is a self-hosted alternative

> Tuning methods (Grid, Random, Optuna) were covered in the Week 7 notebook.